In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

catalog = "sbsrisk_dev"

spark.sql(f"USE CATALOG {catalog}")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

silver_tipo = f"{catalog}.silver.silver_morosidad_tipo_long"
df = spark.table(silver_tipo)

display(df.limit(5))
print("rows silver_tipo:", df.count())

In [0]:
# Mapa de abreviaturas
month_map = F.create_map(
    [F.lit(x) for x in [
        "en",1,"fe",2,"ma",3,"ab",4,"my",5,"jn",6,"jl",7,"ag",8,"se",9,"oc",10,"no",11,"di",12
    ]]
)

df_gold_base = (
    df
    .withColumn("periodo_mes", F.lower(F.col("periodo_mes")))
    .withColumn("mes", month_map[F.col("periodo_mes")].cast("int"))
    .withColumn("fecha", F.last_day(F.to_date(F.concat_ws("-", F.col("anio"), F.lpad(F.col("mes"),2,"0"), F.lit("01")))))
    .withColumn("cmac", F.trim(F.col("cmac")))
    .withColumn("concepto", F.trim(F.col("concepto")))
    .withColumn("valor", F.col("valor").cast("double"))
    .filter(F.col("fecha").isNotNull())
    .filter(F.col("valor").isNotNull())
)

display(df_gold_base.orderBy("fecha","concepto","cmac").limit(30))
print("rows gold_base:", df_gold_base.count())

In [0]:
gold_ts = f"{catalog}.gold.gold_morosidad_tipo_timeseries"

(df_gold_base
 .select("fecha","anio","mes","periodo_mes","cmac","concepto","valor","source_file")
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .saveAsTable(gold_ts)
)

spark.sql(f"SELECT COUNT(*) AS total FROM {gold_ts}").show()

In [0]:
from pyspark.sql.window import Window

N = 5

w_desc = Window.partitionBy("fecha","concepto").orderBy(F.col("valor").desc(), F.col("cmac"))
w_asc  = Window.partitionBy("fecha","concepto").orderBy(F.col("valor").asc(),  F.col("cmac"))

df_rank = (
    df_gold_base
    .select("fecha","anio","mes","periodo_mes","concepto","cmac","valor")
    .withColumn("rank_desc", F.row_number().over(w_desc))
    .withColumn("rank_asc",  F.row_number().over(w_asc))
    .withColumn(
        "ranking_tipo",
        F.when(F.col("rank_desc") <= N, F.lit("TOP"))
         .when(F.col("rank_asc")  <= N, F.lit("BOTTOM"))
    )
    .filter(F.col("ranking_tipo").isNotNull())
    .withColumn("rank", F.when(F.col("ranking_tipo")=="TOP", F.col("rank_desc")).otherwise(F.col("rank_asc")))
    .drop("rank_desc","rank_asc")
)

gold_rank = f"{catalog}.gold.gold_morosidad_tipo_ranking"

(df_rank
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .saveAsTable(gold_rank)
)

spark.sql(f"SELECT ranking_tipo, COUNT(*) filas FROM {gold_rank} GROUP BY ranking_tipo").show()
display(spark.table(gold_rank).orderBy("fecha","concepto","ranking_tipo","rank").limit(50))

In [0]:
import re

gold_pivot = f"{catalog}.gold.gold_morosidad_tipo_pivot"

df_pivot = (
    df_gold_base
    .groupBy("fecha","anio","mes","periodo_mes","cmac")
    .pivot("concepto")
    .agg(F.first("valor"))
)

clean_cols = [re.sub(r'[ ,;{}()\n\t=*]', '_', c) for c in df_pivot.columns]
df_pivot = df_pivot.toDF(*clean_cols)

(df_pivot
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema","true")
 .saveAsTable(gold_pivot)
)

spark.sql(f"SELECT COUNT(*) AS total FROM {gold_pivot}").show()

# Validaciones

In [0]:
%sql
SELECT MIN(fecha) min_fecha, MAX(fecha) max_fecha, COUNT(*) filas
FROM sbsrisk_dev.gold.gold_morosidad_tipo_timeseries;

SELECT fecha, concepto, COUNT(*) cmac_distintas
FROM sbsrisk_dev.gold.gold_morosidad_tipo_timeseries
GROUP BY fecha, concepto
ORDER BY fecha, concepto;

In [0]:
%sql
SELECT 
  MIN(fecha) AS min_fecha,
  MAX(fecha) AS max_fecha,
  COUNT(*)   AS filas
FROM sbsrisk_dev.gold.gold_morosidad_tipo_timeseries;

In [0]:
%sql
SELECT 
  COUNT(DISTINCT fecha) AS meses,
  COUNT(DISTINCT cmac)  AS cmac,
  COUNT(*)              AS filas
FROM sbsrisk_dev.gold.gold_morosidad_tipo_pivot;